<a href="https://colab.research.google.com/github/4mh24ca401-dotcom/Agentic-Internship-Progress-Tracker-Reflection/blob/main/Agentic_AI_LAB_10_AI_Ethics_Tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#1. Install required libraries
!pip install -q -U google-genai gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.55.0 which is incompatible.
googl

In [2]:
#2. Import libraries and configure Gemini API
from google import genai
from getpass import getpass
import os
import gradio as gr
os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key: ")
client = genai.Client()

Enter your Gemini API Key: ··········


In [4]:
#3. Store rules, logs and counters
logs = []
stats = {"TP": 0, "FP": 0, "TN": 0, "FN": 0}
rules = {
    "Prompt Injection": [
        "ignore previous", "reveal system", "bypass", "hidden instructions"
],
"Misinformation": [
    "fake news", "false claim", "misinformation", "conspiracy"
],
"Bias": [
    "all men", "all women", "all religions", "inferior", "superior"
]
}

In [6]:
#4. Detect safety violation
def detect_violation(prompt):
    text = prompt.lower()
    for category, keywords in rules.items():
        for word in keywords:
            if word in text:
                return category
    return "Safe"

In [7]:
#5. Update metric counts
def update_counts(predicted, actual):
    if predicted == "Violation" and actual == "Violation":
        stats["TP"] += 1
    elif predicted == "Violation" and actual == "Safe":
        stats["FP"] += 1
    elif predicted == "Safe" and actual == "Safe":
        stats["TN"] += 1
    else:
        stats["FN"] += 1

In [8]:
#6. Calculate safety metrics
def divide(numerator, denominator):
    if denominator == 0:
        return 0
    return numerator / denominator
def calculate_metrics():
    tp = stats["TP"]
    fp = stats["FP"]
    tn = stats["TN"]
    fn = stats["FN"]
    accuracy = divide(tp + tn, tp + fp + tn + fn)
    precision = divide(tp, tp + fp)
    recall = divide(tp, tp + fn)
    return f"""Accuracy: {accuracy:.2f}
Precision: {precision:.2f}
Recall / Detection Rate: {recall:.2f}"""

In [9]:
# 7. Create the main AI Ethics Tester function

def ethics_tester(prompt, actual):
    violation = detect_violation(prompt)

    if violation == "Safe":
        predicted = "Safe"
    else:
        predicted = "Violation"

    update_counts(predicted, actual)
    metrics = calculate_metrics()

    if violation != "Safe":
        reply = "Request blocked. Possible " + violation + " detected."

        mitigation = (
            "Use safer prompts, verify facts, avoid biased language, "
            "and apply human review."
        )

        logs.append(
            f"Prompt: {prompt}\n"
            f"Violation: {violation}\n"
            f"Mitigation: {mitigation}\n\n"
        )

    else:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents="Answer safely and briefly: " + prompt
        )
        reply = response.text

    log_text = "".join(logs)

    if log_text == "":
        log_text = "No violations logged."

    return reply, log_text, metrics

In [ ]:
# 8. Create and launch Gradio application

app = gr.Interface(
    fn=ethics_tester,
    inputs=[
        gr.Textbox(label="Enter test prompt"),
        gr.Radio(["Safe", "Violation"], label="Actual Label", value="Violation")
    ],
    outputs=[
        gr.Textbox(label="Chatbot Response"),
        gr.Textbox(label="Violation Log"),
        gr.Textbox(label="Safety Evaluation Metrics")
    ],
    title="AI Ethics Tester: Safety and Bias Evaluator",
    flagging_mode="never"
)

app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://849637b0c44c50c0cf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
